# P7 (abreviada): modelo bigrama y conexión con el decoder

En este notebook verificamos los cálculos principales de la práctica:

1. Conteos de unigramas y bigramas;
2. Estimación MV y suavizado add-$\alpha$;
3. Probabilidad y perplejidad de una oración;
4. Combinación de score acústico y score de lenguaje.

In [1]:
import numpy as np
from collections import Counter, defaultdict

## 1. Corpus y vocabulario

In [2]:
corpus = [
    ['<s>', 'el', 'gato', 'come', 'pescado', '</s>'],
    ['<s>', 'el', 'perro', 'come', 'carne', '</s>'],
    ['<s>', 'el', 'gato', 'come', 'carne', '</s>'],
    ['<s>', 'el', 'perro', 'corre', '</s>'],
    ['<s>', 'el', 'gato', 'corre', '</s>'],
]

vocab = sorted({w for oracion in corpus for w in oracion})
V = len(vocab)
print(f'Vocabulario ({V}): {vocab}')

Vocabulario (9): ['</s>', '<s>', 'carne', 'come', 'corre', 'el', 'gato', 'perro', 'pescado']


## 2. Conteos

`unigram[w]` representa $C(w)$ y `bigram[u][v]` representa $C(u,v)$.

In [3]:
def contar_bigramas(corpus):
    unigram = Counter()
    bigram = defaultdict(Counter)

    for oracion in corpus:
        for w in oracion:
            unigram[w] += 1

        for w_prev, w_next in zip(oracion[:-1], oracion[1:]):
            bigram[w_prev][w_next] += 1

    return unigram, bigram


unigram, bigram = contar_bigramas(corpus)
print('C("come") =', unigram['come'])
print('C("come", "pescado") =', bigram['come']['pescado'])
print('C("gato", "perro") =', bigram['gato']['perro'])

C("come") = 3
C("come", "pescado") = 1
C("gato", "perro") = 0


## 3. MV y add-$\alpha$

Para $\alpha=1$, add-$\alpha$ coincide con add-one.

In [4]:
def P_mv(w_next, w_prev, unigram, bigram):
    if unigram[w_prev] == 0:
        return 0.0
    return bigram[w_prev][w_next] / unigram[w_prev]


def P_addalpha(w_next, w_prev, unigram, bigram, vocab, alpha=1.0):
    V = len(vocab)
    return (bigram[w_prev][w_next] + alpha) / (unigram[w_prev] + alpha * V)


for w_next, w_prev in [('pescado', 'come'), ('perro', 'gato')]:
    print(f'P("{w_next}" | "{w_prev}")')
    print(f'MV: {P_mv(w_next, w_prev, unigram, bigram):.4f}')
    print(f'add-one: {P_addalpha(w_next, w_prev, unigram, bigram, vocab, alpha=1.0):.4f}')

P("pescado" | "come")
MV: 0.3333
add-one: 0.1667
P("perro" | "gato")
MV: 0.0000
add-one: 0.0833


## 4. Probabilidad y perplejidad

In [5]:
def log_prob_oracion(oracion, unigram, bigram, vocab, metodo='addalpha', alpha=1.0):
    logp = 0.0
    n_pred = 0

    for w_prev, w_next in zip(oracion[:-1], oracion[1:]):
        if metodo == 'mv':
            p = P_mv(w_next, w_prev, unigram, bigram)
        elif metodo == 'addalpha':
            p = P_addalpha(w_next, w_prev, unigram, bigram, vocab, alpha)
        else:
            raise ValueError('Método desconocido')

        if p == 0:
            return -np.inf, n_pred

        logp += np.log(p)
        n_pred += 1

    return logp, n_pred


def perplejidad(oracion, unigram, bigram, vocab, metodo='addalpha', alpha=1.0):
    logp, n_pred = log_prob_oracion(oracion, unigram, bigram, vocab, metodo, alpha)
    if logp == -np.inf:
        return np.inf
    return np.exp(-logp / n_pred)


oracion = ['<s>', 'el', 'gato', 'come', 'carne', '</s>']
logp, n_pred = log_prob_oracion(oracion, unigram, bigram, vocab, alpha=1.0)
print(f'log P = {logp:.4f}')
print(f'P = {np.exp(logp):.6f}')
print(f'N_pred = {n_pred}')
print(f'PP = {perplejidad(oracion, unigram, bigram, vocab, alpha=1.0):.4f}')

log P = -6.1719
P = 0.002087
N_pred = 5
PP = 3.4363


## 5. Efecto de $\alpha$

Comparar la probabilidad de un bigrama observado, la de uno no observado y la perplejidad de la misma oración.

In [6]:
alphas = [1.0, 0.5, 0.1, 0.01]

print(f'{"alpha":>8s}  P("pescado"|"come")  P("perro"|"gato")  PP(oracion)')
for alpha in alphas:
    p_visto = P_addalpha('pescado', 'come', unigram, bigram, vocab, alpha)
    p_nuevo = P_addalpha('perro', 'gato', unigram, bigram, vocab, alpha)
    pp = perplejidad(oracion, unigram, bigram, vocab, alpha=alpha)
    print(f'{alpha:8.2f}  {p_visto:20.4f}  {p_nuevo:18.4f}  {pp:11.3f}')

   alpha  P("pescado"|"come")  P("perro"|"gato")  PP(oracion)
    1.00                0.1667              0.0833        3.436
    0.50                0.2000              0.0667        2.559
    0.10                0.2821              0.0256        1.600
    0.01                0.3269              0.0032        1.334


Al disminuir $\alpha$, la probabilidad del bigrama observado se acerca a su valor por MV y el bigrama no observado recibe menos masa. La perplejidad de esta oración de entrenamiento también disminuye, pero esto no permite elegir $\alpha$: debería ajustarse con datos de validación.

## 6. Conexión con el decoder

Después del prefijo `"el"`, se comparan `"gato"` y `"perro"`. Los valores de `delta_s_ac` son incrementos de score acústico. Usar el resultado del ejercicio escrito para identificar qué términos determinan la comparación.

In [7]:
candidatas = np.array(['gato', 'perro'])
delta_s_ac = np.array([-10.5, -10.3])
p_lm = np.array([0.6, 0.4])
eta = 1.0

delta_s_comparacion = delta_s_ac + eta * np.log(p_lm)
eta_c = (delta_s_ac[1] - delta_s_ac[0]) / np.log(p_lm[0] / p_lm[1])

print(dict(zip(candidatas, delta_s_comparacion)))
print('Decisión:', candidatas[np.argmax(delta_s_comparacion)])
print(f'eta_c = {eta_c:.4f}')

{np.str_('gato'): np.float64(-11.01082562376599), np.str_('perro'): np.float64(-11.216290731874155)}
Decisión: gato
eta_c = 0.4933


### Interpretación

- La acústica sola favorece `"perro"`, porque $-10{,}3>-10{,}5$.
- Para $\eta=1$, el score combinado favorece `"gato"`.
- Hay empate para $\eta=\eta_c$ y se elige `"gato"` si $\eta>\eta_c$ (con $\eta_c=0{,}4933$).
- Las dos hipótesis comparten el mismo prefijo y agregan una palabra. Por eso reciben el mismo término de inserción, el cual se cancela al comparar los scores.
